# Drug design with tree ensembles

*This notebook requires a full Gurobi license and manually supplied Merck
training data (see below). It is not run by `make` in the `notebooks`
directory.*

We train a random forest to predict molecular activity, formulate descriptor
optimization with `"misic"`, solve it, and inspect the solution. Next we add
the paper's shared-leaf constraints and inspect their effect. Only the last
section compares alternative formulations.

We revisit the case study of Mišić, *Optimization of Tree Ensembles*,
Operations Research 68(5):1605–1624, 2020
([doi:10.1287/opre.2019.1928](https://doi.org/10.1287/opre.2019.1928)),
adapted to scikit-learn rather than R's `randomForest`. Optimized descriptors
do not describe a realizable molecule.

This notebook is self-contained. No other notebook needs to be run first.

In [ ]:
import time

import gurobipy as gp
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.ensemble import RandomForestRegressor

from gurobi_ml import add_predictor_constr

pd.set_option("display.float_format", "{:.3f}".format)

## Data and application

In its case study (Section 6), Mišić (2020) maximizes the predicted activity
of a molecule with random forests of 100 trees, trained with a third of the
features considered at each split, at least five samples per leaf, and no
depth limit, on the Merck molecular activity data released by Ma et al.,
*Deep Neural Nets as a Method for Quantitative Structure–Activity
Relationships*, Journal of Chemical Information and Modeling 55(2):263–274,
2015 ([doi:10.1021/ci500747n](https://doi.org/10.1021/ci500747n)).

The data is not downloaded automatically. To run this notebook, download the
Supporting Information of Ma et al. (2015) from the article page linked above
(check its terms of use), unzip it, and put `METAB_training_disguised.csv` in
the same directory as this notebook. `METAB` is one of the smallest datasets
(1569 molecules with 4372 descriptors).

In [ ]:
frame = pd.read_csv("METAB_training_disguised.csv")
frame.shape

## Train the molecular activity predictor

The `Act` column is the activity to predict, and the descriptors are the
columns after the first two metadata columns. To keep this example
manageable, we use 20 trees of depth at most 5, rather than the paper's 100
unlimited-depth trees. Each split considers a third of the features, and each
leaf contains at least five samples. A fixed random seed makes training
repeatable.

The optimization remains an illustration, not a full reproduction or a
high-accuracy molecular activity model.

In [ ]:
X_drug = frame.iloc[:, 2:].astype(float)
y_drug = frame["Act"].astype(float)
drug_forest = RandomForestRegressor(
    n_estimators=20,
    max_depth=5,
    max_features=1 / 3,
    min_samples_leaf=5,
    random_state=0,
).fit(X_drug.to_numpy(), y_drug.to_numpy())

## Model descriptor optimization with one formulation

The decision vector $x$ contains molecular descriptors. We maximize the
forest's predicted activity subject to bounds from the training data:

$$
\max_x f(x) \qquad
\text{subject to }\min_i X_{ij} \le x_j \le \max_i X_{ij}
\quad\text{for each descriptor }j.
$$

`add_predictor_constr` links the descriptors to the prediction variable used
as the objective. We use `"misic"`, the paper's formulation, with the default
`epsilon=0`. Its split decisions are shared across trees.

`solve` takes an optional `add_constraints` hook, which is unused in
the first solve and adds the shared-leaf constraints in the next section.

In [ ]:
def solve(predictor, X, formulation="misic", add_constraints=None):
    """Build and solve the model; return a dict of results and the solution."""
    start = time.perf_counter()
    model = gp.Model()
    model.Params.OutputFlag = 0
    model.Params.Seed = 0
    x = model.addMVar(
        (1, X.shape[1]),
        lb=X.min().to_numpy(),
        ub=X.max().to_numpy(),
        name="descriptors",
    )
    # epsilon=0 lets the solver put features exactly on split thresholds and take
    # either branch there, so the objective can exceed what the forest predicts
    # at x (see "prediction at x" in the results). A positive epsilon removes
    # this ambiguity, but see the discussion after the first solve.
    prediction = add_predictor_constr(
        model, predictor, x, formulation=formulation, epsilon=0.0
    )
    if add_constraints is not None:
        add_constraints(model, prediction)
    model.setObjective(prediction.output.sum(), gp.GRB.MAXIMIZE)
    build_time = time.perf_counter() - start
    model.optimize()

    # Without an incumbent, only the status and the bound are available.
    solved = model.SolCount > 0
    nan = float("nan")
    result = {
        "formulation": formulation,
        "variables": model.NumVars,
        "binaries": model.NumBinVars,
        "linear constraints": model.NumConstrs,
        "general constraints": model.NumGenConstrs,
        "build [s]": build_time,
        "solve [s]": model.Runtime,
        "status": model.Status,
        "objective": model.ObjVal if solved else nan,
        "bound": model.ObjBound,
        "gap": model.MIPGap if solved else nan,
        "prediction at x": predictor.predict(x.X)[0] if solved else nan,
    }
    solution = (
        pd.Series(x.X[0], index=X.columns, name="descriptors") if solved else None
    )
    return result, solution


def nonzero(solution):
    """The nonzero entries of a solution, if there is one."""
    return None if solution is None else solution[solution != 0]

## Solve and inspect the unconstrained descriptor solution

`solve` uses a fixed solver seed. The first table reports the status, the
best bound and the gap, since an incumbent of a solve that stops early is not
necessarily optimal, as well as the forest's prediction at the solution. The
second shows the nonzero descriptors of the solution. If the solver finds no
incumbent, the objective is empty and no descriptor vector is returned.

In [ ]:
baseline_result, baseline_solution = solve(drug_forest, X_drug)
display(pd.Series(baseline_result))
display(nonzero(baseline_solution))

The solution is an optimized descriptor vector, not a molecule, and its
activity is a prediction, not a measurement or a claim of safety or efficacy.

**The objective is optimistic.** With `epsilon=0`, an input exactly on a split
threshold satisfies both branches, and the solver is free to place descriptors
on thresholds and take the better branch. The forest's `predict` must choose
one branch, and it rounds inputs to single precision, so `prediction at x` can
be well below the `objective` (compare them in the table above).

A positive `epsilon` removes the ambiguity, but it must be larger than
Gurobi's `IntFeasTol` (1e-5 by default) and than the single-precision
resolution of the feature, otherwise `predict` rounds the input back onto the
threshold. It can also make the model infeasible. Reading the chosen leaves
from `tree_leaves` avoids recomputing the prediction altogether.

See [Tree Ensemble Formulations](../tree-ensembles/Tree%20Ensemble%20Formulations.ipynb)
for selected-leaf checks before interpreting this as achievable predicted
activity.

## Constrain shared leaves with training molecules

For each training molecule, constraint (11) of Mišić (2020) caps the fraction
of trees in which the optimized descriptor vector selects that molecule's
leaf:

$$
\sum_{t=1}^{T} z_{t,\ell_t(x_i)} \le cT
\qquad\text{for each training molecule }i.
$$

Here `z` is a selected-leaf variable and `ell_t(x_i)` is the leaf reached by
training molecule `i` in tree `t`. This is an **upper bound on shared leaves**,
not a guarantee that a solution is close to the training data or that it is
a realizable molecule.

Every formulation compared below exposes `tree_leaves`, so the same linear
constraints can be added regardless of the chosen formulation. Unreachable leaves have no
variable and contribute zero.

In [ ]:
def proximity_constraints(forest, X, c):
    """Mišić (2020), constraint (11): for every training point, at most a
    fraction c of the trees send the solution to the leaf of that point."""
    leaf_of = forest.apply(X.to_numpy())  # node reached by each point in each tree

    def add(model, prediction):
        total = 0
        for t, (leaf_vars, nodes) in enumerate(prediction.tree_leaves):
            # Column of each node in leaf_vars. Leaves that no input can
            # reach have no variable (column -1) and count as 0.
            column = np.full(forest.estimators_[t].tree_.node_count, -1)
            column[np.asarray(nodes, dtype=int)] = np.arange(len(nodes))
            cols = column[leaf_of[:, t]]
            rows = np.flatnonzero(cols >= 0)
            indicator = sparse.csr_matrix(
                (np.ones(len(rows)), (rows, cols[rows])), shape=(len(X), len(nodes))
            )
            total = total + indicator @ leaf_vars[0, :]
        model.addConstr(total <= c * len(prediction.tree_leaves))

    return add

## Solve again with the shared-leaf restriction

We keep the same forest, descriptor bounds, and `"misic"` formulation, and
set `c = 0.1`: at most 10% of the trees, here 2 of 20, may send the solution
to the leaf of any given training molecule. The value is illustrative; a
smaller `c` tightens the restriction. The table puts the baseline and the
constrained solve side by side, followed by the constrained descriptor
vector.

The restriction is not guaranteed to be feasible. If the problem is
infeasible or no incumbent is found, the status shows it, the objective is
empty and no descriptor vector is returned.

In [ ]:
constrained_result, constrained_solution = solve(
    drug_forest, X_drug, add_constraints=proximity_constraints(drug_forest, X_drug, 0.1)
)
display(pd.DataFrame([baseline_result, constrained_result], index=["none", "c = 0.1"]))
display(nonzero(constrained_solution))

## Compare formulations

We reuse the same 20-tree, depth-5 forest, with and without the shared-leaf
restriction (`c = 0.1`), and run each of the five formulations
(see [Decision Trees and Tree Ensembles](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html#decision-trees-and-tree-ensembles) in the user guide). Only the
formulation changes, and no new forest is trained.

The table reports the size of each model, the time to build it and to solve
it, and the solution status, objective, bound and gap.

**Runtime:** the solves have no time limit, and constructing the models and
the shared-leaf constraints takes noticeable time with thousands of
descriptors. With these small forests the ten solves took about ten seconds in
our run, but larger forests (such as the paper's 100 unlimited-depth trees) or
several seeds can take substantially longer.

In [ ]:
FORMULATIONS = ["leaf", "misic", "misic_lazy", "ocean", "biggs_perakis"]

rows = []
for c in (None, 0.1):
    label = "none" if c is None else f"c = {c}"
    constraints = None if c is None else proximity_constraints(drug_forest, X_drug, c)
    for formulation in FORMULATIONS:
        print(f"proximity {label}, {formulation}", flush=True)
        result, _ = solve(drug_forest, X_drug, formulation, constraints)
        rows.append({"proximity": label, **result})
display(pd.DataFrame(rows))

Compare the `none` and `c = 0.1` rows: adding the restriction can only lower
the optimal objective, so the difference is the cost of requiring the
solution to avoid the leaves of training molecules. Also check whether the
formulations handle the extra constraints equally well, and compare
`prediction at x` with `objective`: they can differ considerably.

Inspect `status`, `bound`, and `gap`, not only solve times. A missing
objective means no incumbent was found. With `epsilon=0`, incompatible leaves
in some formulations can also explain objective differences; an objective
alone does not establish an achievable prediction.

The [concrete and wine example](../tree-ensembles/Concrete%20and%20Wine%20Forest%20Optimization.ipynb)
uses the same application-first workflow without these leaf constraints.
The documentation's
[small concrete example](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/auto_examples/example5_concrete_strength.html)
can be run with a size-limited license.

Copyright © 2023-2026 Gurobi Optimization, LLC